In [5]:
import pandas as pd

path = "../data/accepted_2007_to_2018Q4.csv.gz"
df = pd.read_csv(path, nrows=1000, low_memory=False)

print(df.shape)
print(df.columns.tolist())

(1000, 151)
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_

In [6]:
cols = ['id', 'issue_d', 'loan_status', 'term', 'loan_amnt']
df2 = pd.read_csv(path, usecols=cols, low_memory=False)

print(df2.shape)
print(df2['loan_status'].value_counts(dropna=False))
print(df2['issue_d'].str[-4:].value_counts().sort_index())

(2260701, 5)
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64
issue_d
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134814
2014    235629
2015    421095
2016    434407
2017    443579
2018    495242
Name: count, dtype: int64


In [7]:
# 1) 비정상 행 확인
print(df2[df2['issue_d'].isna()].head())

# 2) 비정상 행 제거 후 발행 연도 만들기
df2 = df2.dropna(subset=['issue_d']).copy()
df2['issue_year'] = df2['issue_d'].str[-4:].astype(int)

# 3) 만기 × 발행 연도별 주요 상태 건수
ct = pd.crosstab([df2['term'], df2['issue_year']], df2['loan_status'])
ct[['Fully Paid', 'Charged Off', 'Current', 'Late (31-120 days)']]

                                                      id  loan_amnt term  \
421095  Total amount funded in policy code 1: 6417608175        NaN  NaN   
421096  Total amount funded in policy code 2: 1944088810        NaN  NaN   
528961  Total amount funded in policy code 1: 1741781700        NaN  NaN   
528962   Total amount funded in policy code 2: 564202131        NaN  NaN   
651664  Total amount funded in policy code 1: 1791201400        NaN  NaN   

       issue_d loan_status  
421095     NaN         NaN  
421096     NaN         NaN  
528961     NaN         NaN  
528962     NaN         NaN  
651664     NaN         NaN  


loan_status           Fully Paid  Charged Off  Current  Late (31-120 days)
term      issue_year                                                      
36 months 2007               206           45        0                   0
          2008              1315          247        0                   0
          2009              4122          594        0                   0
          2010              7624          842        0                   0
          2011             12602         1499        0                   0
          2012             37567         5903        0                   0
          2013             88044        12378        0                   0
          2014            140255        22315        0                   0
          2015            240894        42132       72                  61
          2016            186241        46112    86623                2831
          2017            102817        25721   183950                5101
          2018             35804         5464   296050                4481
60 months 2010              2425          645        0                   0
          2011              5822         1798        0                   0
          2012              7156         2741        0                   0
          2013             25736         8646        6                   3
          2014             41686        18846    11919                 334
          2015             58848        33671    43227                1298
          2016             38612        22130    47438                1715
          2017             27335        13427    77901                2903
          2018             11640         3403   131131                2740

In [8]:
resolved = ct['Fully Paid'] + ct['Charged Off']
unresolved = (ct['Current'] + ct['In Grace Period']
              + ct['Late (16-30 days)'] + ct['Late (31-120 days)'])

summary = pd.DataFrame({
    '확정 건수': resolved,
    '미확정 건수': unresolved,
    '미확정 비율(%)': (unresolved / (resolved + unresolved) * 100).round(1),
    '부실률(확정 기준,%)': (ct['Charged Off'] / resolved * 100).round(1),
})
summary

확정 건수  미확정 건수  미확정 비율(%)  부실률(확정 기준,%)
term      issue_year                                         
36 months 2007           251       0        0.0          17.9
          2008          1562       0        0.0          15.8
          2009          4716       0        0.0          12.6
          2010          8466       0        0.0           9.9
          2011         14101       0        0.0          10.6
          2012         43470       0        0.0          13.6
          2013        100422       0        0.0          12.3
          2014        162570       0        0.0          13.7
          2015        283026     147        0.1          14.9
          2016        232353   91134       28.2          19.8
          2017        128538  191868       59.9          20.0
          2018         41268  303399       88.0          13.2
60 months 2010          3070       0        0.0          21.0
          2011          7620       0        0.0          23.6
          2012          9897       0        0.0          27.7
          2013         34382      10        0.0          25.1
          2014         60532   12526       17.1          31.1
          2015         92519   45402       32.9          36.4
          2016         60742   50168       45.2          36.4
          2017         40762   82390       66.9          32.9
          2018         15043  135525       90.0          22.6

In [9]:
scope = df2[(df2['term'].str.strip() == '36 months')
            & (df2['issue_year'].between(2012, 2015))]

print(scope.shape)
print(scope['loan_status'].value_counts(dropna=False))
print(pd.crosstab(scope['issue_year'], scope['loan_status']))

(589635, 6)
loan_status
Fully Paid            506760
Charged Off            82728
Current                   72
Late (31-120 days)        61
In Grace Period            9
Late (16-30 days)          5
Name: count, dtype: int64
loan_status  Charged Off  Current  Fully Paid  In Grace Period  \
issue_year                                                       
2012                5903        0       37567                0   
2013               12378        0       88044                0   
2014               22315        0      140255                0   
2015               42132       72      240894                9   

loan_status  Late (16-30 days)  Late (31-120 days)  
issue_year                                          
2012                         0                   0  
2013                         0                   0  
2014                         0                   0  
2015                         5                  61  
